# Data Information

This dataset contains insurance cost information for 25,000 applicants from various cities in India. It includes demographic, health, and lifestyle variables such as age, gender, BMI, cholesterol level, smoking status, exercise habits, alcohol consumption, and more. The target variable is `insurance_cost`, which represents the insurance premium charged to the individual.

# Dataset

In [ ]:
import pandas as pd

data_df = pd.read_csv("Data.csv")
data_df.head(10)

# Problem Definition

## Objective

- Use the dataset to estimate insurance cost per applicant
- Identify which variables are most associated with higher or lower insurance cost

## Usecase

Corporate Wellness Program
- Identify high-risk employees based on predicted insurance costs
- Enable targeted interventions to reduce the company's overall healthcare expenses

Insurance Company
- Analyze insurance cost trends across different population segments
- Enable dynamic premium pricing based on individual risk profiles
- Provide personalized insurance plans tailored to each customer

Example Use Case
An applicant with certain health conditions and lifestyle factors who is predicted to have high insurance costs can be recommended lifestyle interventions or appropriate coverage plans

# Data Preparation

## 1. Data Understanding

In [ ]:
print("Shape:", data_df.shape)

In [ ]:
print("Columns:", data_df.columns.tolist())

In [ ]:
data_df.info()

In [ ]:
data_df.isnull().sum()

In [ ]:
data_df.duplicated().sum()

In [ ]:
data_df = data_df.drop_duplicates()
data_df.duplicated().sum()

In [ ]:
data_df.describe().round(2)

In [ ]:
data_df.describe(include="object")

# 2. Data Cleaning

## 2.1 Drop ID Column

In [ ]:
data_df = data_df.drop(columns=['applicant_id'])
print("Shape after dropping applicant_id:", data_df.shape)

## 2.2 Missing Values

In [ ]:
print("Missing values:")
missing = data_df.isnull().sum()
print(missing[missing > 0])
print(f"\nbmi missing: {data_df['bmi'].isnull().sum()} ({data_df['bmi'].isnull().mean()*100:.1f}%)")
print(f"Year_last_admitted missing: {data_df['Year_last_admitted'].isnull().sum()} ({data_df['Year_last_admitted'].isnull().mean()*100:.1f}%)")

In [ ]:
data_df['bmi'] = data_df['bmi'].fillna(data_df['bmi'].median())
print(f"bmi missing after imputation: {data_df['bmi'].isnull().sum()}")

In [ ]:
data_df['was_admitted'] = data_df['Year_last_admitted'].notna().astype(int)
data_df = data_df.drop(columns=['Year_last_admitted'])
print("was_admitted value counts:")
print(data_df['was_admitted'].value_counts())

In [ ]:
print("Remaining missing values:", data_df.isnull().sum().sum())

## 2.3 Outlier

In [ ]:
num_cols = ['age', 'bmi', 'avg_glucose_level', 'daily_avg_steps', 'weight', 'fat_percentage', 'insurance_cost']

for col in num_cols:
    Q1 = data_df[col].quantile(0.25)
    Q3 = data_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = data_df[(data_df[col] < lower) | (data_df[col] > upper)]
    
    print(f"{col}: {len(outliers)} outliers (lower={lower:.1f}, upper={upper:.1f})")

Outliers in numeric features are retained as they represent real-world variability in health metrics.

Outlier insurance_cost are retained because they are the target variable that the model must predict.

In [ ]:
data_df['Occupation'].value_counts()

In [ ]:
data_df['cholesterol_level'].value_counts()

In [ ]:
data_df['Gender'].value_counts()

In [ ]:
data_df['smoking_status'].value_counts()

In [ ]:
data_df['Location'].value_counts()

In [ ]:
data_df['Alcohol'].value_counts()

In [ ]:
data_df['exercise'].value_counts()

In [ ]:
data_df['covered_by_any_other_company'].value_counts()

# 3. Data Transformation

## 3.1 Feature Encoding

In [ ]:
data_df['gender_encoded'] = data_df['Gender'].map({'Male': 0, 'Female': 1})
data_df.head(5)

In [ ]:
data_df['covered_other_encoded'] = data_df['covered_by_any_other_company'].map({'N': 0, 'Y': 1})
data_df.head(5)

In [ ]:
alcohol_map = {'No': 0, 'Rare': 1, 'Daily': 2}
data_df['alcohol_encoded'] = data_df['Alcohol'].map(alcohol_map)

exercise_map = {'No': 0, 'Moderate': 1, 'Extreme': 2}
data_df['exercise_encoded'] = data_df['exercise'].map(exercise_map)

data_df[['Alcohol', 'alcohol_encoded', 'exercise', 'exercise_encoded']].head()

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe_cols = ['Occupation', 'cholesterol_level', 'smoking_status', 'Location']
encoder = OneHotEncoder(sparse_output=False, drop='first')

encoded_arr = encoder.fit_transform(data_df[ohe_cols])
encoded_columns = encoder.get_feature_names_out(ohe_cols)
encoded_df = pd.DataFrame(encoded_arr, columns=encoded_columns, index=data_df.index)

data_df = data_df.drop(columns=ohe_cols)
data_df = pd.concat([data_df, encoded_df], axis=1)

print("Shape after encoding:", data_df.shape)
data_df.head(5)

In [ ]:
data_df = data_df.drop(columns=['Gender', 'covered_by_any_other_company', 'Alcohol', 'exercise'])
print("Final shape:", data_df.shape)
data_df.head(5)

# Data Analysis

## 1. Descriptive Statistics

In [ ]:
data_df[['age', 'bmi', 'avg_glucose_level', 'daily_avg_steps', 'weight', 'fat_percentage', 'insurance_cost']].describe().round(2)

## 2. Inferential Statistics

## 2.1 Normality Test

In [ ]:
from scipy import stats

stat, p = stats.shapiro(data_df['insurance_cost'].sample(5000, random_state=42))
print(f"Shapiro-Wilk (sample 5000): stat={stat:.4f}, p-value={p:.6f}")
print(f"Skewness: {data_df['insurance_cost'].skew():.2f}")

The Shapiro-Wilk test rejects normality (p < 0.05), so we use non-parametric tests. A sample of 5000 is used because Shapiro-Wilk has a maximum input size limit.

## 2.2 Binary Features VS Insurance Cost (Mann-Whitney U Test)

In [ ]:
from scipy.stats import mannwhitneyu

binary_cols = ['gender_encoded', 'covered_other_encoded', 'adventure_sports', 'heart_decs_history', 'other_major_decs_history', 'was_admitted']

for col in binary_cols:
    group_0 = data_df[data_df[col] == 0]['insurance_cost']
    group_1 = data_df[data_df[col] == 1]['insurance_cost']
    stat, p = mannwhitneyu(group_0, group_1, alternative='two-sided')
    print(f"{col}: U={stat:.2f}, p-value={p:.6f} --> {'Significant' if p < 0.05 else 'Not Significant'}")

## 2.3 Multi-group Features vs Insurance Cost (Kruskal-Wallis Test)

In [ ]:
from scipy.stats import kruskal

plot_df_temp = pd.read_csv("Data.csv").drop_duplicates()

multi_group_cols = ['Occupation', 'cholesterol_level', 'smoking_status', 'Location', 'Alcohol', 'exercise']

for col in multi_group_cols:
    groups = [group['insurance_cost'].values for name, group in plot_df_temp.groupby(col)]
    stat, p = kruskal(*groups)
    print(f"{col}: H={stat:.2f}, p-value={p:.6f} --> {'Significant' if p < 0.05 else 'Not Significant'}")

del plot_df_temp

## 2.4 Continuous Features vs Insurance Cost (Spearman Correlation)

In [ ]:
from scipy.stats import spearmanr

continuous_cols = ['age', 'bmi', 'avg_glucose_level', 'daily_avg_steps', 'weight', 'fat_percentage',
                   'years_of_insurance_with_us', 'regular_checkup_lasy_year', 'visited_doctor_last_1_year',
                   'weight_change_in_last_one_year']

for col in continuous_cols:
    corr, p = spearmanr(data_df[col], data_df['insurance_cost'])
    print(f"{col}: rho={corr:.4f}, p-value={p:.6f} --> {'Significant' if p < 0.05 else 'Not Significant'}")

## 3. Correlation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

core_cols = ['age', 'bmi', 'avg_glucose_level', 'daily_avg_steps', 'weight', 'fat_percentage',
             'years_of_insurance_with_us', 'regular_checkup_lasy_year', 'visited_doctor_last_1_year',
             'weight_change_in_last_one_year', 'gender_encoded', 'covered_other_encoded',
             'adventure_sports', 'heart_decs_history', 'other_major_decs_history',
             'was_admitted', 'alcohol_encoded', 'exercise_encoded', 'insurance_cost']

corr_matrix = data_df[core_cols].corr(method='spearman').round(2)

plt.figure(figsize=(16, 12))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, vmin=-1, vmax=1, fmt='.2f', annot_kws={'size': 7})
plt.title('Spearman Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
cost_corr = corr_matrix['insurance_cost'].drop('insurance_cost').sort_values(ascending=False)
print("Correlation with insurance_cost (Spearman):\n")
print(cost_corr.to_string())

# Data Visualization

In [ ]:
plot_df = pd.read_csv("Data.csv").drop_duplicates()

## 1. Distribution

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(plot_df['insurance_cost'], bins=40, edgecolor='black', alpha=0.7)
plt.axvline(plot_df['insurance_cost'].mean(), color='red', linestyle='--', label=f"Mean: {plot_df['insurance_cost'].mean():,.0f}")
plt.axvline(plot_df['insurance_cost'].median(), color='orange', linestyle='--', label=f"Median: {plot_df['insurance_cost'].median():,.0f}")
plt.xlabel('Insurance Cost')
plt.ylabel('Frequency')
plt.title('Distribution of Insurance Cost')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = ['age', 'bmi', 'avg_glucose_level', 'daily_avg_steps', 'weight', 'fat_percentage']
categorical_cols = ['Gender', 'Occupation', 'smoking_status', 'Alcohol', 'exercise', 'covered_by_any_other_company']

In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(plot_df[col].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.show()

In [ ]:
for col in categorical_cols:
    plt.figure(figsize=(10, 5))
    ax = sns.countplot(data=plot_df, x=col, order=plot_df[col].value_counts().index)
    plt.title(f"Count Plot of {col}")
    plt.xticks(rotation=45)
    
    for container in ax.containers:
        ax.bar_label(container)
    
    plt.tight_layout()
    plt.show()

## 2. Insurance Cost by Smoking Status

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=plot_df, x='smoking_status', y='insurance_cost',
            order=['never smoked', 'formerly smoked', 'smokes', 'Unknown'])
plt.title('Insurance Cost by Smoking Status')
plt.xlabel('Smoking Status')
plt.ylabel('Insurance Cost')
plt.tight_layout()
plt.show()

## 3. Age vs Insurance Cost

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=plot_df, x='age', y='insurance_cost', hue='Gender', alpha=0.3,
                palette={'Male': '#3498db', 'Female': '#e74c3c'})
sns.regplot(data=plot_df, x='age', y='insurance_cost', scatter=False, color='red')
plt.title('Age vs Insurance Cost (colored by Gender)')
plt.xlabel('Age')
plt.ylabel('Insurance Cost')
plt.legend(title='Gender')
plt.tight_layout()
plt.show()

## 4. BMI vs Insurance Cost

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=plot_df, x='bmi', y='insurance_cost', hue='Gender', alpha=0.3,
                palette={'Male': '#3498db', 'Female': '#e74c3c'})
plt.title('BMI vs Insurance Cost (colored by Gender)')
plt.xlabel('BMI')
plt.ylabel('Insurance Cost')
plt.legend(title='Gender')
plt.tight_layout()
plt.show()

## 5. Insurance Cost by Occupation

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=plot_df, x='Occupation', y='insurance_cost',
            order=['Student', 'Salried', 'Business'])
plt.title('Insurance Cost by Occupation')
plt.xlabel('Occupation')
plt.ylabel('Insurance Cost')
plt.tight_layout()
plt.show()

## 6. Insurance Cost by Gender and Alcohol Consumption

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=plot_df, x='Gender', y='insurance_cost', order=['Male', 'Female'], ax=axes[0])
axes[0].set_title('Insurance Cost by Gender')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Insurance Cost')

sns.boxplot(data=plot_df, x='Alcohol', y='insurance_cost', order=['No', 'Rare', 'Daily'], ax=axes[1])
axes[1].set_title('Insurance Cost by Alcohol Consumption')
axes[1].set_xlabel('Alcohol')
axes[1].set_ylabel('Insurance Cost')

plt.tight_layout()
plt.show()